In [0]:
from pytickersymbols import PyTickerSymbols
import yfinance as yf
from pyspark.sql.types import StructType, StructField, DateType, DoubleType, LongType, StringType, BooleanType
from pyspark.sql.functions import max as spark_max
import pandas as pd
import pandas_market_calendars as mcal
from datetime import datetime

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS yfinance_pipeline_dev;
USE CATALOG yfinance_pipeline_dev;
CREATE SCHEMA IF NOT EXISTS stocks_dataset;
USE SCHEMA stocks_dataset;

In [0]:
stock_data = PyTickerSymbols()

dowjones_stocks = stock_data.get_stocks_by_index('DOW JONES')
dowjones_symbols = [stock['symbol'] for stock in dowjones_stocks]

In [0]:
collected_symbols = []

# for symbol_list in [nasdaq100_symbols, dowjones_symbols, sp500_symbols, sp600_symbols]:
for symbol in dowjones_symbols:
    if symbol not in collected_symbols and "." not in symbol:
        collected_symbols.append(symbol)

print(f"Total unique symbols: {len(collected_symbols)}")
print(collected_symbols[:20])  # Show first 20 for preview

In [0]:
# sample_tickers = yf.Tickers("AAPL MSFT GOOGL GOOG AMZN META TSLA NVDA")

In [0]:
def load_stocks_price(
    collected_symbols,
    table_name,
    fetch_period,
    batch_size=100
):
    schema = StructType([
        StructField("symbol", StringType(), True),
        StructField("date", DateType(), True),
        StructField("open", DoubleType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("volume", LongType(), True)
    ])

    # 3. Processing in loops
    tickers = yf.Tickers(collected_symbols)
    # Initialize the Tickers object for this batch
    dfs = []
    # Example: Accessing data for each ticker in the current batch
    for symbol in collected_symbols:
        try:
            # Note: .info is slow. If you only need prices, use .history instead.
            # info_data = tickers_obj.tickers[symbol].info

            hist = tickers.tickers[symbol].history(period=fetch_period).reset_index()
            hist['symbol'] = symbol
            hist = hist[['symbol', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
            hist.columns = ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']
            hist['date'] = hist['date'].dt.date
            dfs.append(hist)

        except Exception as e:
            print(f"Error fetching {symbol}: {e}")

    stocks_hist = pd.concat(dfs, ignore_index=True)


    stocks_hist_spark = spark.createDataFrame(stocks_hist, schema=schema)

    # Create or replace table stocks_price_history
    stocks_hist_spark.write.mode("append").saveAsTable(table_name)

    return 

In [0]:
nyse = mcal.get_calendar('NYSE')
today = datetime.now().date()

# Get schedule for the current month up to today
schedule = nyse.schedule(start_date='2026-01-01', end_date=today)
last_trading_day = schedule.index[-1].date()
print(f"Current Date: {today}")
print(f"Last Trading Day: {last_trading_day}")

In [0]:
def update_price_table(table_name, symbols, target_date):
    """
    Checks if the table is up to date and loads data if necessary.
    """
    # if not spark.catalog.tableExists(table_name):
    #     print(f"Table {table_name} does not exist. Initializing load.")
    #     load_stocks_price(
    #         collected_symbols=symbols,
    #         table_name=table_name,
    #         fetch_period="1d"
    #     )
    #     print(f"Complete: {table_name}")
    #     return

    # Get the latest date currently in the table
    df = spark.table(table_name)
    # Assumes 'date' is the column name
    latest_db_date = df.select(spark_max("date")).collect()[0][0]

    if latest_db_date < target_date:
        print(f"Updating {table_name}. DB date ({latest_db_date}) < Last trading day ({target_date}).")
        load_stocks_price(
            collected_symbols=symbols,
            table_name=table_name,
            fetch_period="1d"
        )
        print(f"Complete: {table_name}")
    else:
        print(f"Skipping {table_name}. Already up to date ({latest_db_date}).")


In [0]:
# --- Execution ---
# Assumes 'last_trading_day' and 'collected_symbols' are already defined
update_price_table(
    table_name="stocks_price_history_raw",
    symbols=collected_symbols,
    target_date=last_trading_day
)

update_price_table(
    table_name="indexes_price_history",
    symbols=["^GSPC", "^DJI", "^NDX"],
    target_date=last_trading_day
)